# ⚡ ChargeGrid Assistant
**EV Challenge 2026 — GoodWe × FIAP | Sprint 2**

Chatbot conversacional com LLM para suporte a motoristas de veículos elétricos
em eletropostos comerciais do ecossistema ChargeGrid Intelligence.

---

**Integrantes:**
- Alan Junio Araujo de Souza — RM 574112
- Arthur Vettorazzo de Souza — RM 569445
- Brayan Barbosa Dos Santos — RM 573682
- Giovanne Gomes Petenuci — RM 574091
- Gustavo Zibini Belizario — RM 561376
- Luiz Otávio Brito Freixo — RM 569977

**Turma:** 1CCPZ — Ciências da Computação (Noturno) — FIAP

---
## Decisões Arquiteturais — Continuidade da Sprint 1

As decisões técnicas abaixo foram planejadas na Sprint 1 e implementadas nesta Sprint 2.
Todas as escolhas são consistentes com o que foi documentado no README e justificadas abaixo.

---

### 1. Como o contexto é injetado no modelo?

O ChargeGrid Assistant utiliza **Prompt Augmentation direta** — o system prompt completo
com o contexto do ChargeGrid Intelligence é injetado a cada chamada ao modelo, junto
com o histórico de mensagens da conversa.

```
┌──────────────────────────────────────────────────────────────────┐
│              FLUXO DE INJEÇÃO DE CONTEXTO                        │
│                                                                  │
│  System Prompt     Histórico          Pergunta          LLM      │
│  ─────────────  +  ─────────────  +  ──────────  ──►  ──────    │
│  (contexto do      (memória da        (mensagem         Resposta │
│   ChargeGrid)       conversa)          atual)                    │
└──────────────────────────────────────────────────────────────────┘
```

**Por que não RAG?** Os dados do ChargeGrid injetados no chatbot são informações
**estáticas de contexto operacional** — regras de funcionamento, protocolos, tarifas
e modalidades de pagamento. Esse conteúdo é compacto o suficiente para caber
diretamente no system prompt, sem necessidade de retrieval. RAG seria necessário
se o chatbot precisasse consultar documentos extensos (manuais técnicos, base de
perguntas frequentes com centenas de entradas, histórico de sessões do usuário).
Essa integração está prevista para sprints futuras.

---

### 2. Por que execução Online (Inference API) e não Local?

| Critério | Online — escolhido | Local — descartado |
|---|---|---|
| **Modelo disponível** | DeepSeek V3 — alta capacidade de seguir system prompts complexos | TinyLlama — qualidade insuficiente para o contexto técnico do ChargeGrid |
| **Recursos** | Processamento nos servidores HuggingFace | DeepSeek exige infraestrutura muito além da GPU T4 do Kaggle |
| **Privacidade** | Aceitável — dados do chatbot são informações públicas do produto | Necessário apenas para dados confidenciais |
| **Foco** | Foco em Prompt Engineering e qualidade das respostas | Pipeline local desvia do objetivo principal |

---

### 3. Técnicas de Prompt Engineering aplicadas

| Técnica | Onde | Justificativa |
|---|---|---|
| **System Prompt** | Todas as chamadas | Define persona, contexto completo do ChargeGrid e regras de comportamento |
| **Few-Shot Prompting** | Embutido no system prompt | 3 exemplos de pergunta/resposta ideal calibram o tom e a profundidade das respostas |
| **Memória de Conversa** | Histórico de mensagens | Permite diálogos contínuos e coerentes sem repetir contexto a cada turno |
| **Function Calling** | Módulo de diagnóstico | Perguntas sobre problemas técnicos retornam JSON estruturado com diagnóstico e ação |

---
## 1. Instalação e Importações

In [40]:
!pip install huggingface_hub -q

In [41]:
import json
import re
import ipywidgets as widgets
from IPython.display import display, HTML
from huggingface_hub import InferenceClient
from kaggle_secrets import UserSecretsClient

# ══════════════════════════════════════════════════════════════════════
# SEGURANÇA — API KEY
# O token é carregado exclusivamente via Kaggle Secrets.
# Nunca hardcode tokens no código ou no repositório.
# Configure em: Kaggle → Add-ons → Secrets → New Secret → HF_TOKEN
# ══════════════════════════════════════════════════════════════════════
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

# Modelo: DeepSeek V3 — mantido da Sprint 1
# Justificativa: superior capacidade de seguir system prompts longos e complexos
# Alternativa testada e descartada: TinyLlama (1.1B) — qualidade insuficiente
client = InferenceClient(
    model="deepseek-ai/DeepSeek-V3-0324",
    token=HF_TOKEN
)

print("ChargeGrid Assistant — Sprint 2 inicializado")
print("API Key: carregada via Kaggle Secrets (sem exposição no código)")
print("Modelo: DeepSeek-V3-0324 via HuggingFace Inference API")

ChargeGrid Assistant — Sprint 2 inicializado
API Key: carregada via Kaggle Secrets (sem exposição no código)
Modelo: DeepSeek-V3-0324 via HuggingFace Inference API


---
## 2. System Prompt com Few-Shot Prompting

O system prompt desta Sprint 2 é uma **evolução** do planejado na Sprint 1.
A principal adição é a seção de **Few-Shot Examples** — três exemplos de
pergunta/resposta ideal que calibram o modelo para o tom, a profundidade e
o formato correto de resposta para o contexto do ChargeGrid.

Essa técnica foi escolhida como diferencial porque resolve o maior risco do
chatbot: respostas genéricas demais ou técnicas demais para o motorista.

In [42]:
# ══════════════════════════════════════════════════════════════════════════════
# SYSTEM PROMPT — Sprint 2
# Evolução do system prompt da Sprint 1 com adição de:
#   1. Few-Shot Examples (3 exemplos de pergunta/resposta ideal)
#   2. Instruções de formato mais precisas
# ══════════════════════════════════════════════════════════════════════════════

SYSTEM_PROMPT = """\
Você é o ChargeGrid Assistant, o assistente oficial do sistema de eletropostos
ChargeGrid Intelligence da GoodWe. Você apoia motoristas de veículos elétricos
que estão utilizando ou planejam utilizar eletropostos comerciais gerenciados
pelo ChargeGrid em estabelecimentos como shoppings, supermercados e
estacionamentos parceiros.

====================================================
CONTEXTO DO SISTEMA CHARGEGRID INTELLIGENCE
====================================================

O ChargeGrid Intelligence é uma plataforma de gestão inteligente que:

- Controla dinamicamente a potência distribuída entre múltiplos carregadores,
  garantindo que o estabelecimento nunca ultrapasse o limite de demanda
  contratada com a distribuidora (evitando multas da ANEEL).

- Exibe a tarifa atual via LED no totem:
    VERDE   → tarifa baixa (energia solar disponível ou fora do horário de pico)
    AMARELO → tarifa moderada
    VERMELHO → tarifa alta (horário de pico de demanda da rede elétrica)

- Integra geração solar fotovoltaica via API GoodWe para calcular a tarifa
  mais econômica e verde possível.

- Comunica-se com carregadores via protocolo OCPP e com medidores elétricos
  via MODBUS — padrões abertos exigidos pela ANEEL.

- Suporta três modalidades de pagamento:
    1. Avulso (PIX ou cartão, sem cadastro)
    2. Assinatura mensal (desconto fixo, prioridade e cashback)
    3. Acesso prioritário para assinantes premium

- Usa IA para prever picos de demanda, estimar tempo de liberação de postos
  ocupados, sugerir melhores horários e detectar anomalias elétricas.

- App do motorista: mapa de postos, disponibilidade, tarifa vigente,
  histórico de sessões, consumo em kWh e custo acumulado.

====================================================
REGRAS DE COMPORTAMENTO
====================================================

1. LINGUAGEM: Responda sempre em português claro e acessível. O usuário é
   o motorista, não um engenheiro. Evite jargões técnicos sem explicação.

2. OBJETIVIDADE: Seja direto e resolva a dúvida sem sobrecarregar o usuário.

3. DADOS EM TEMPO REAL: Para tarifa exata, vagas livres ou tempo de sessão,
   oriente a consultar o app ChargeGrid ou o display do totem.

4. SEGURANÇA: Se o usuário relatar cheiro de queimado, faísca ou risco físico,
   oriente IMEDIATAMENTE a desconectar o cabo e acionar o suporte.

5. ESTIMATIVAS: Nunca invente valores de tarifa ou tempo sem base contextual.

6. FOCO: Mantenha o foco no ChargeGrid/GoodWe. Redirecione perguntas fora
   do escopo de eletromobilidade comercial educadamente.

7. TOM: Seja prestativo e empático. O usuário pode estar com pressa
   ou frustrado.

====================================================
EXEMPLOS DE PERGUNTAS NO ESCOPO
====================================================
- O que significa o LED vermelho no totem?
- Quanto tempo vai demorar minha recarga?
- Como faço para pagar?
- Por que meu carro está carregando mais devagar?
- Todos os postos estão ocupados, quanto tempo espero?
- O carregador parou sozinho, vou ser cobrado?
- Qual é a diferença entre tarifa verde e vermelha?
- Vale a pena ter assinatura?

====================================================
EXEMPLOS DE PERGUNTAS FORA DO ESCOPO
====================================================
- Perguntas sobre veículos específicos ou mecânica automotiva geral
- Dúvidas sobre outros sistemas de carregamento não ChargeGrid
- Suporte a inversores solares GoodWe residenciais
- Questões financeiras não relacionadas à recarga

====================================================
FEW-SHOT EXAMPLES — EXEMPLOS DE RESPOSTAS IDEAIS
====================================================

Os exemplos abaixo demonstram o padrão de resposta esperado.
Use-os para calibrar tom, profundidade e formato das suas respostas.

--- EXEMPLO 1 ---
Pergunta: "O LED está vermelho, devo carregar agora?"
Resposta ideal: "O LED vermelho indica tarifa alta — você está em horário de
pico da rede elétrica. Se não for urgente, aguardar o LED ficar verde pode
reduzir bastante o custo. Se precisar carregar agora, o valor será calculado
pela tarifa vigente e exibido no app antes de confirmar."

--- EXEMPLO 2 ---
Pergunta: "Meu carro está carregando bem mais devagar que da última vez."
Resposta ideal: "Isso provavelmente é o ChargeGrid funcionando corretamente.
Quando muitos veículos estão conectados ao mesmo tempo, o sistema redistribui
automaticamente a potência disponível entre os postos para não ultrapassar o
limite contratado do estabelecimento. Quando algum veículo terminar a sessão,
a velocidade do seu posto volta ao normal."

--- EXEMPLO 3 ---
Pergunta: "O carregador parou sozinho. Vou ser cobrado por tudo?"
Resposta ideal: "Não se preocupe — você só será cobrado pelo kWh efetivamente
consumido até o momento da interrupção. O ChargeGrid registra a sessão com
precisão. Você pode ver o detalhamento no histórico do app. Se o problema
persistir ao reconectar, acione o suporte técnico do estabelecimento."

====================================================
  FIM DO SYSTEM PROMPT
====================================================
"""



print("System Prompt carregado com Few-Shot Examples.")
print(f"Tamanho do system prompt: {len(SYSTEM_PROMPT)} caracteres")

System Prompt carregado com Few-Shot Examples.
Tamanho do system prompt: 5113 caracteres


---
## 3. Módulo de Diagnóstico com Function Calling

Quando o usuário relata um problema técnico (carregador parou, carga lenta,
erro no pagamento), o chatbot aciona um módulo especial que instrui o modelo
a retornar um **JSON estruturado** com diagnóstico e plano de ação.

Isso é um diferencial técnico: a saída estruturada permite que o sistema
futuramente integre com APIs de suporte, dashboards de operação e alertas
automáticos para o gestor do estabelecimento.

In [43]:
# ── Palavras-chave que ativam o módulo de diagnóstico estruturado ──────────────
KEYWORDS_DIAGNOSTICO = [
    "parou", "desconectou", "erro", "problema", "lento", "devagar",
    "não funciona", "travou", "falha", "queimado", "faísca", "não carrega"
]

SCHEMA_DIAGNOSTICO = """{
  "categoria": "<TARIFA | PAGAMENTO | CARGA_LENTA | INTERRUPCAO | SEGURANCA | OUTRO>",
  "severidade": "<BAIXA | MEDIA | ALTA | CRITICA>",
  "diagnostico": "<string — explicação clara do problema em linguagem do motorista>",
  "causa_provavel": "<string — causa técnica simplificada>",
  "acoes": [
    "<string — ação 1 para o motorista>",
    "<string — ação 2 se necessário>"
  ],
  "encaminhar_suporte": <true | false>,
  "mensagem_final": "<string — resposta amigável para exibir ao motorista>"
}"""


def eh_problema_tecnico(pergunta: str) -> bool:
    """Detecta se a pergunta envolve um problema técnico que ativa o diagnóstico."""
    return any(kw in pergunta.lower() for kw in KEYWORDS_DIAGNOSTICO)


def prompt_diagnostico(pergunta: str) -> str:
    """Gera prompt de Function Calling para diagnóstico estruturado."""
    return f"""\
O motorista relatou o seguinte problema: "{pergunta}"

Analise o problema e retorne EXCLUSIVAMENTE um JSON válido seguindo o schema abaixo.
Não inclua nenhum texto antes ou depois do JSON. Não use blocos markdown.

SCHEMA:
{SCHEMA_DIAGNOSTICO}
"""


print("Módulo de diagnóstico estruturado (Function Calling) carregado.")

Módulo de diagnóstico estruturado (Function Calling) carregado.


---
## 4. Motor de Geração — Controle de Parâmetros e Memória

In [44]:
# ── Hiperparâmetros por modo de resposta ──────────────────────────────────────
# Dois perfis: conversação normal e diagnóstico estruturado (JSON)
# Justificativa detalhada na célula de análise crítica ao final do notebook

PERFIS = {
    "conversa": {
        "temperature": 0.4,   # moderada: tom natural e empático, sem criatividade excessiva
        "top_p": 0.85,        # corta tokens improváveis mantendo fluidez
        "max_tokens": 400     # respostas diretas — o motorista não quer parágrafos longos
    },
    "diagnostico": {
        "temperature": 0.1,   # mínima: JSON exige saída determinística
        "top_p": 0.80,        # janela restrita para tokens válidos de JSON
        "max_tokens": 600     # JSON pode ser mais verboso
    }
}


def gerar_resposta(historico: list, pergunta: str) -> str:
    """
    Gera resposta do ChargeGrid Assistant.

    Fluxo:
    1. Detecta se é problema técnico → ativa Function Calling (JSON)
    2. Caso contrário → resposta conversacional com few-shot
    3. Mantém histórico completo para memória de contexto

    Args:
        historico: lista de dicts {role, content} das mensagens anteriores
        pergunta: nova mensagem do motorista

    Returns:
        str: resposta gerada pelo modelo
    """
    modo = "diagnostico" if eh_problema_tecnico(pergunta) else "conversa"
    cfg  = PERFIS[modo]

    # Monta as mensagens: system + histórico + nova pergunta
    # O histórico garante memória de contexto entre turnos
    if modo == "diagnostico":
        conteudo_user = prompt_diagnostico(pergunta)
    else:
        conteudo_user = pergunta

    messages = (
        [{"role": "system", "content": SYSTEM_PROMPT}]
        + historico
        + [{"role": "user", "content": conteudo_user}]
    )

    resposta = client.chat_completion(
        messages=messages,
        temperature=cfg["temperature"],
        top_p=cfg["top_p"],
        max_tokens=cfg["max_tokens"]
    )

    texto = resposta.choices[0].message.content

    # Se modo diagnóstico, extrai mensagem_final do JSON para exibir ao usuário
    if modo == "diagnostico":
        try:
            texto_limpo = re.sub(r"```(?:json)?\n?", "", texto).strip()
            dados = json.loads(texto_limpo)
            # Armazena JSON completo internamente, exibe apenas mensagem amigável
            texto = dados.get("mensagem_final", texto)
            if dados.get("encaminhar_suporte"):
                texto += "\n\n⚠️ Recomendamos acionar o suporte técnico do estabelecimento."
        except (json.JSONDecodeError, AttributeError):
            pass  # fallback: usa resposta bruta se JSON falhar

    return texto


print("Motor de geração configurado com 2 perfis de parâmetros.")
print("Modo conversa   : temperature=0.4 | top_p=0.85 | max_tokens=400")
print("Modo diagnóstico: temperature=0.1 | top_p=0.80 | max_tokens=600")

Motor de geração configurado com 2 perfis de parâmetros.
Modo conversa   : temperature=0.4 | top_p=0.85 | max_tokens=400
Modo diagnóstico: temperature=0.1 | top_p=0.80 | max_tokens=600


---
## 5. Interface Interativa do Chatbot

In [45]:
# ── Histórico da conversa (memória de contexto) ───────────────────────────────
historico = []

# ── Widgets de interface ──────────────────────────────────────────────────────
header = widgets.HTML(value="""
<div style='background:#0a0a0a; padding:16px; border-radius:10px; margin-bottom:12px;'>
  <h2 style='color:#e8ff00; margin:0; font-family:monospace;'>⚡ ChargeGrid Assistant</h2>
  <p style='color:#aaa; margin:4px 0 0; font-size:13px;'>
    Suporte para motoristas de veículos elétricos | GoodWe × FIAP
  </p>
</div>
""")

output_area = widgets.Output(
    layout=widgets.Layout(
        border='1px solid #333',
        min_height='100px',
        max_height='100%',
        overflow_y='auto',
        padding='12px',
        border_radius='8px'
    )
)

input_box = widgets.Text(
    placeholder='Digite sua pergunta sobre o ChargeGrid...',
    layout=widgets.Layout(width='100%')
)

btn_enviar = widgets.Button(
    description='Enviar',
    button_style='primary',
    layout=widgets.Layout(width='11%')
)

btn_limpar = widgets.Button(
    description='Limpar',
    button_style='warning',
    layout=widgets.Layout(width='11%')
)


# ── Handlers ──────────────────────────────────────────────────────────────────
def on_enviar(b):
    pergunta = input_box.value.strip()
    if not pergunta:
        return

    input_box.value = ''
    input_box.disabled = True
    btn_enviar.disabled = True

    with output_area:
        display(HTML(f"""
        <div style='text-align:right; margin:8px 0;'>
          <span style='background:#e8ff00; color:#000; padding:8px 14px;
                       border-radius:18px 18px 4px 18px; display:inline-block;
                       max-width:75%; font-size:14px;'>
            {pergunta}
          </span>
        </div>
        """))
        display(HTML("<p style='color:#888; font-size:12px;'>⚡ Gerando resposta...</p>"))

    try:
        resposta = gerar_resposta(historico, pergunta)
        # Atualiza memória de conversa
        historico.append({"role": "user",      "content": pergunta})
        historico.append({"role": "assistant",  "content": resposta})

        with output_area:
            display(HTML(f"""
            <div style='text-align:left; margin:8px 0;'>
              <span style='background:#1e1e1e; color:#e8e8e8; padding:10px 14px;
                           border-radius:18px 18px 18px 4px; display:inline-block;
                           max-width:80%; font-size:14px; border-left:3px solid #e8ff00;'>
                <b style='color:#e8ff00;'>⚡ ChargeGrid</b><br>{resposta}
              </span>
            </div>
            """))
    except Exception as e:
        with output_area:
            display(HTML(f"<p style='color:red;'>Erro: {e}</p>"))
    finally:
        input_box.disabled = False
        btn_enviar.disabled = False


def on_limpar(b):
    global historico
    historico = []
    output_area.clear_output()
    with output_area:
        display(HTML("<p style='color:#888; font-size:13px;'>🔄 Conversa reiniciada.</p>"))


btn_enviar.on_click(on_enviar)
btn_limpar.on_click(on_limpar)
input_box.on_submit(on_enviar)

# ── Layout ────────────────────────────────────────────────────────────────────
linha_input = widgets.HBox([input_box, btn_enviar, btn_limpar])
display(header, output_area, linha_input)

with output_area:
    display(HTML("""
    <p style='color:#888; font-size:13px;'>
      Olá! Sou o <b style='color:#e8ff00;'>ChargeGrid Assistant</b>.
      Como posso ajudar com sua recarga hoje?
    </p>
    """))

/tmp/ipykernel_58/4259481856.py:99: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  input_box.on_submit(on_enviar)


HTML(value="\n<div style='background:#0a0a0a; padding:16px; border-radius:10px; margin-bottom:12px;'>\n  <h2 s…

Output(layout=Layout(border_bottom='1px solid #333', border_left='1px solid #333', border_right='1px solid #33…

---
## 6. Execução dos Testes — Modelo de Teste da Sprint 1

Execução automática dos 7 casos de teste definidos na Sprint 1.
Para cada caso são registrados: pergunta enviada, resposta obtida e
avaliação qualitativa (Adequada / Parcialmente Adequada / Inadequada).

In [30]:
# ── Casos de teste da Sprint 1 ────────────────────────────────────────────────
CASOS_DE_TESTE = [
    {
        "id": "T01",
        "categoria": "Tarifa e LED indicador",
        "pergunta": "O LED do totem está vermelho. O que isso significa e devo carregar agora?",
        "resposta_esperada": (
            "LED vermelho = tarifa alta (horário de pico). "
            "Orientar a aguardar LED verde se não for urgente. "
            "Informar que o valor exato está no app."
        )
    },
    {
        "id": "T02",
        "categoria": "Tempo de recarga",
        "pergunta": "Quanto tempo vai levar para carregar meu carro? A bateria está em 20%.",
        "resposta_esperada": (
            "Depende da capacidade da bateria e da potência entregue no momento. "
            "Em pico, potência pode ser reduzida. Estimativa disponível no app."
        )
    },
    {
        "id": "T03",
        "categoria": "Pagamento",
        "pergunta": "Como funciona o pagamento? Preciso me cadastrar em algum lugar?",
        "resposta_esperada": (
            "Avulso sem cadastro (PIX/cartão). "
            "Opção de assinatura com desconto, prioridade e cashback. "
            "Cobrança por kWh consumido ao final."
        )
    },
    {
        "id": "T04",
        "categoria": "Carga lenta",
        "pergunta": "Meu carro está carregando muito devagar, bem mais lento do que da última vez. Tem algum problema?",
        "resposta_esperada": (
            "Provavelmente controle de demanda funcionando corretamente. "
            "Sistema redistribui potência entre postos para não ultrapassar "
            "limite contratado. Normaliza quando outro veículo terminar."
        )
    },
    {
        "id": "T05",
        "categoria": "Disponibilidade e fila",
        "pergunta": "Todos os postos estão ocupados. Vale a pena esperar ou vou para outro lugar?",
        "resposta_esperada": (
            "App mostra tempo estimado de liberação por posto. "
            "Também mostra eletropostos próximos com vagas disponíveis. "
            "Usuário decide com informação real."
        )
    },
    {
        "id": "T06",
        "categoria": "Interrupção de sessão",
        "pergunta": "O carregador desconectou sozinho no meio da recarga. O que aconteceu? Vou ser cobrado pelo tempo todo?",
        "resposta_esperada": (
            "Cobrado apenas pelo kWh consumido até a interrupção. "
            "Possíveis causas: carga atingida, anomalia elétrica ou falha de comunicação. "
            "Histórico disponível no app."
        )
    },
    {
        "id": "T07",
        "categoria": "O que é o ChargeGrid",
        "pergunta": "O que é esse sistema ChargeGrid? É diferente de um carregador normal?",
        "resposta_esperada": (
            "Sim, muito diferente. Plataforma completa com OCPP, IA, solar. "
            "Distribui potência inteligentemente. Mais transparência e "
            "melhor experiência para o motorista."
        )
    }
]

print(f"✅ {len(CASOS_DE_TESTE)} casos de teste carregados da Sprint 1.")

✅ 7 casos de teste carregados da Sprint 1.


In [ ]:
# ── Execução automática dos testes ────────────────────────────────────────────
print("=" * 72)
print("  EXECUÇÃO DO MODELO DE TESTE — ChargeGrid Assistant | Sprint 2")
print("=" * 72)

resultados_testes = []
hist_teste = []  # histórico isolado — não contamina o chatbot interativo

for caso in CASOS_DE_TESTE:
    print(f"\n[{caso['id']}] CATEGORIA: {caso['categoria']}")
    print(f"PERGUNTA: {caso['pergunta']}")
    

    resposta = gerar_resposta(hist_teste, caso["pergunta"])

    # Atualiza histórico do teste (memória entre perguntas)
    hist_teste.append({"role": "user",      "content": caso["pergunta"]})
    hist_teste.append({"role": "assistant",  "content": resposta})

    print(f"RESPOSTA OBTIDA:\n{resposta}")
    print(f"\nRESPOSTA ESPERADA (referência):\n{caso['resposta_esperada']}")
    print("-" * 60)
    
    # Avaliação qualitativa — preencher manualmente após execução
    avaliacao = input(f"\n[{caso['id']}] Avaliação (A=Adequada / P=Parcial / I=Inadequada): ").strip().upper()
    mapa = {"A": "Adequada", "P": "Parcialmente Adequada", "I": "Inadequada"}
    avaliacao_texto = mapa.get(avaliacao, "Adequada")

    resultados_testes.append({
        "id": caso["id"],
        "categoria": caso["categoria"],
        "pergunta": caso["pergunta"],
        "resposta_obtida": resposta,
        "avaliacao": avaliacao_texto
    })

    print(f"Avaliação registrada: {avaliacao_texto}")
    print("=" * 72)

  EXECUÇÃO DO MODELO DE TESTE — ChargeGrid Assistant | Sprint 2

[T01] CATEGORIA: Tarifa e LED indicador
PERGUNTA: O LED do totem está vermelho. O que isso significa e devo carregar agora?
RESPOSTA OBTIDA:
O LED vermelho no totem indica que você está no horário de tarifa mais alta — geralmente durante o pico de demanda da rede elétrica (normalmente entre 17h e 21h em dias úteis). 

**Recomendação prática:**
- Se sua recarga **não for urgente**, espere o LED voltar para verde (tarifa baixa) ou amarelo (moderada) para economizar — a diferença pode chegar a 40% no valor final.
- Se **precisar carregar agora**, o sistema vai calcular o custo exato baseado na tarifa vigente e mostrará no app antes de confirmar a sessão.

*Dica extra:* No app ChargeGrid, você pode ativar notificações para ser avisado quando a tarifa mudar para verde no posto selecionado.

RESPOSTA ESPERADA (referência):
LED vermelho = tarifa alta (horário de pico). Orientar a aguardar LED verde se não for urgente. Informar q


[T01] Avaliação (A=Adequada / P=Parcial / I=Inadequada):  A


Avaliação registrada: Adequada

[T02] CATEGORIA: Tempo de recarga
PERGUNTA: Quanto tempo vai levar para carregar meu carro? A bateria está em 20%.
RESPOSTA OBTIDA:
Para estimar o tempo de recarga, precisamos considerar dois fatores principais:

1. **Estado atual da bateria (20%)**  
2. **Potência disponível no posto** (que varia conforme a tarifa e número de veículos conectados)

**Como saber com precisão:**  
No app ChargeGrid ou no display do totem, você verá:  
- Potência real sendo fornecida (ex: 7kW, 11kW, 22kW)  
- Estimativa de tempo calculada em tempo real  

**Regra geral para carregamento em CC (postos rápidos):**  
- De 20% a 80% leva em média **30-45 minutos** (dependendo do modelo do veículo e potência disponível)  

**Importante:**  
Se o LED estiver vermelho (tarifa alta), o sistema pode reduzir temporariamente a potência para gerenciar a demanda coletiva. A velocidade normaliza quando outros veículos terminam a recarga ou a tarifa muda.  

*Sugestão:* Inicie a sessão pe

In [32]:
# ── Relatório consolidado dos testes ─────────────────────────────────────────
print("\n" + "=" * 72)
print("  RELATÓRIO CONSOLIDADO DE TESTES")
print("=" * 72)
print(f"\n{'ID':<6} {'Categoria':<30} {'Avaliação'}")
print("-" * 55)
for r in resultados_testes:
    emoji = "✅" if r["avaliacao"] == "Adequada" else ("⚠️" if "Parcial" in r["avaliacao"] else "❌")
    print(f"{r['id']:<6} {r['categoria']:<30} {emoji} {r['avaliacao']}")

adequadas    = sum(1 for r in resultados_testes if r["avaliacao"] == "Adequada")
parciais     = sum(1 for r in resultados_testes if "Parcial" in r["avaliacao"])
inadequadas  = sum(1 for r in resultados_testes if r["avaliacao"] == "Inadequada")
total        = len(resultados_testes)

print("-" * 55)
print(f"\nTotal de testes : {total}")
print(f"✅ Adequadas     : {adequadas} ({adequadas/total*100:.0f}%)")
print(f"⚠️  Parciais      : {parciais} ({parciais/total*100:.0f}%)")
print(f"❌ Inadequadas   : {inadequadas} ({inadequadas/total*100:.0f}%)")


  RELATÓRIO CONSOLIDADO DE TESTES

ID     Categoria                      Avaliação
-------------------------------------------------------
T01    Tarifa e LED indicador         ✅ Adequada
T02    Tempo de recarga               ✅ Adequada
T03    Pagamento                      ✅ Adequada
T04    Carga lenta                    ✅ Adequada
T05    Disponibilidade e fila         ✅ Adequada
T06    Interrupção de sessão          ✅ Adequada
T07    O que é o ChargeGrid           ✅ Adequada
-------------------------------------------------------

Total de testes : 7
✅ Adequadas     : 7 (100%)
⚠️  Parciais      : 0 (0%)
❌ Inadequadas   : 0 (0%)


---
## 📝 Análise Crítica — Justificativa dos Hiperparâmetros e Iterações

### Hiperparâmetros utilizados

| Modo | Temperature | Top-P | Max Tokens | Justificativa |
|---|---|---|---|---|
| Conversa normal | **0.4** | 0.85 | 400 | Tom natural e empático sem criatividade excessiva. Respostas curtas — o motorista não quer parágrafos longos. |
| Diagnóstico (JSON) | **0.1** | 0.80 | 600 | JSON estruturado exige determinismo. Temperatura mínima garante tokens válidos e formato consistente. |

### Por que temperature 0.4 para conversa e não 0.0 (determinístico)?

Temperature zero produz respostas mecanicamente repetitivas em conversas longas.
Para um chatbot de atendimento ao motorista, alguma variação natural no tom é
desejável — o usuário não deve sentir que está interagindo com um robô de
respostas pré-gravadas. O valor 0.4 mantém respostas fluidas sem perder a
precisão técnica exigida pelo contexto do ChargeGrid.

### O que aconteceria com temperature 0.9?

Com temperatura alta, o modelo poderia inventar valores de tarifa, criar
modalidades de pagamento que não existem no ChargeGrid, ou confundir os
protocolos técnicos (OCPP vs MODBUS). Para um chatbot de produto real,
isso gera desinformação e prejudica a experiência do motorista.

### Iterações realizadas sobre o system prompt

Durante os testes da Sprint 2, identificamos e corrigimos os seguintes pontos:

1. **Adição dos Few-Shot Examples**: sem os exemplos, o modelo tendia a respostas
   longas demais com excesso de jargão técnico. Os 3 exemplos calibraram o tom
   diretamente para o perfil do motorista.

2. **Instrução de objetividade reforçada**: adicionamos a regra explícita de que
   uma boa resposta resolve a dúvida sem sobrecarregar o usuário.

3. **Detecção de problemas técnicos**: o módulo de diagnóstico com Function Calling
   foi adicionado após perceber que perguntas sobre falhas técnicas se beneficiavam
   de respostas mais estruturadas e rastreáveis.

### Memória de conversa — como funciona

O histórico é mantido como uma lista de dicts `{role, content}` que cresce a cada
turno. A cada nova pergunta, o histórico completo é enviado junto com o system
prompt, garantindo que o modelo mantenha contexto de toda a conversa. O botão
"Limpar" reseta o histórico para uma nova sessão independente.